# Dokimasia: from raw to bronze

One turbine, two layers. Each layer is profiled, judged against a contract and, if the gate lets the data through,
attested. The bronze layer receives its data through `advance_dataset`, so its profile says where the data came from.

Everything is written to a temporary directory. The last cell checks that nothing else happened: no log output, no
logging configuration, no other files.

In [ ]:
import datetime
import logging
import pathlib
import tempfile

from phoibe import dokimasia
from phoibe.synthetic_data import turbine

WORKDIR = pathlib.Path(tempfile.mkdtemp(prefix="dokimasia-demo-"))
CWD = pathlib.Path.cwd()
CWD_BEFORE = sorted(path.name for path in CWD.iterdir())
ROOT_HANDLERS_BEFORE = list(logging.getLogger().handlers)
SOFTWARE_VERSION = "0.7.0"
NOW = datetime.datetime(2026, 11, 22, 9, 0, tzinfo=datetime.UTC)

## The contract

A contract is a YAML file. This one is written to the temporary directory and loaded back.

In [ ]:
CONTRACT_YAML = """
id: wind-scada-10min
version: 1.0.0
domain: wind
layers:
  raw:
    measures:
      required_signals:
        params: {signals: [power, wind_speed]}
      time_gaps:
        params: {expected_step_s: 600}
    checks:
      signals_present:
        measure: required_signals
        value: n_missing_signals
        severity: critical
        good: {max: 0}
      availability:
        measure: time_gaps
        value: share_missing_steps
        severity: critical
        good: {max: 0.001}
        acceptable: {max: 0.10}
  bronze:
    measures:
      wind_speed_limits:
        rule: ranges
        signal: wind_speed
        params: {min: 0, max: 60}
      power_limits:
        rule: ranges
        signal: power_kw
        params: {min: 0, max: {subject: rated_power_kw, factor: 1.05}}
    checks:
      no_wind_speed_outliers:
        measure: wind_speed_limits
        value: n_out_of_range
        severity: critical
        good: {max: 0}
        acceptable: {max: 10}
      power_within_rating:
        measure: power_limits
        value: n_out_of_range
        severity: warning
        good: {max: 0}
"""
contract_path = WORKDIR / "wind-scada-10min.yaml"
contract_path.write_text(CONTRACT_YAML, encoding="utf-8")
contract = dokimasia.load_contract(contract_path)
[layer.name for layer in contract.layers]

## The data

Generated SCADA data with a short gap. The subject carries what the contract's limits depend on.

In [ ]:
subject = dokimasia.Subject("WEA 01", "wind_turbine", {"rated_power_kw": 5600, "site_timezone": "Europe/Berlin"})
frame = turbine.generate_wtg_scada(time=turbine.Time("2026-02-14T04:00:00", "10min", 432), seed=3)
raw = dokimasia.Dataset.from_frame(frame.drop(frame.index[[40, 41, 42]]))
raw.frame.head(3)

## Which column is which signal

Detection reports what it cannot decide, and never picks one of several candidates.

In [ ]:
detection = dokimasia.wind.detect_wind_signals(raw.signal_names)
detection.raise_if_ambiguous()
detection.build_table()

## Raw layer

Profile and judgment are separate objects; `validate_layer` just runs both.

In [ ]:
raw_validation = dokimasia.validate_layer(raw, subject, contract, "raw")
raw_validation.result

In [ ]:
raw_validation.result.to_frame()[["check_id", "severity", "verdict", "value", "reason"]]

In [ ]:
raw_attestation = dokimasia.attest_validation(raw_validation, created_at=NOW, software_version=SOFTWARE_VERSION)
store = dokimasia.FlatStore(WORKDIR / "runs")
store.save_run(dokimasia.build_run(raw_validation, raw_attestation), created_at=NOW)
raw_attestation

## Advance to bronze

The transform renames the columns to catalog names (the mapping comes from detection) and moves the time axis to
UTC. It refuses where it would have to guess.

In [ ]:
transform = dokimasia.standardize.Chain(
    (
        dokimasia.standardize.RenameToCatalog(detection.mapping),
        dokimasia.standardize.LocalizeTimeToUtc.from_subject(subject),
    )
)
bronze = dokimasia.advance_dataset(raw, raw_attestation, transform, software_version=SOFTWARE_VERSION)
bronze.frame.head(3)

In [ ]:
bronze_validation = dokimasia.validate_layer(bronze, subject, contract, "bronze")
bronze_validation.profile.provenance

In [ ]:
later = NOW + datetime.timedelta(minutes=5)
bronze_attestation = dokimasia.attest_validation(bronze_validation, created_at=later, software_version=SOFTWARE_VERSION)
store.save_run(dokimasia.build_run(bronze_validation, bronze_attestation), created_at=later)
bronze_attestation

## The run index

The index is built from the files, so it cannot go stale. The parent of the bronze run is the raw run.

In [ ]:
entries = store.list_runs()
dokimasia.adapters.build_run_table(entries)[["layer", "created_at", "gate_outcome", "attested"]]

In [ ]:
bronze_entry = dokimasia.adapters.find_entries(entries, layer="bronze")[0]
parent_entry = dokimasia.adapters.find_entries(entries, attestation_id=bronze_entry.parent_id)[0]
reloaded = store.load_run(bronze_entry)

assert reloaded.profile == bronze_validation.profile
assert reloaded.attestation == bronze_attestation
assert store.load_run(parent_entry).attestation == raw_attestation
assert reloaded.profile.provenance.parent_attestation_id == raw_attestation.id
assert reloaded.profile.provenance.fingerprint_matches is True
(parent_entry.layer, "->", bronze_entry.layer)

## Nothing else happened

In [ ]:
assert list(logging.getLogger().handlers) == ROOT_HANDLERS_BEFORE, "the root logger was configured"
assert [type(h) for h in logging.getLogger("phoibe.dokimasia").handlers] == [logging.NullHandler]
assert sorted(path.name for path in CWD.iterdir()) == CWD_BEFORE, "files were written next to the notebook"
sorted(path.name for path in WORKDIR.rglob("*") if path.is_file())